# Semantic Correspondence with Visual Foundation Models
## Stage 1 — Training-Free Baseline

**Pipeline:**
1. Mount Google Drive & unzip SPair-71k
2. Install dependencies
3. Load frozen DINOv2 features
4. For each image pair: cosine similarity → argmax → predicted keypoint
5. Evaluate PCK@{0.05, 0.10, 0.20}

> ⚡ Make sure **Runtime → Change runtime type → T4 GPU** is selected before running.


---
## Cell 1 — Verify GPU


In [ ]:
import torch

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU — go to Runtime -> Change runtime type -> T4 GPU')


---
## Cell 2 — Mount Google Drive


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# ── CHANGE THIS to match where your zip is in Google Drive ────────────────
# Example: '/content/drive/MyDrive/datasets/SPair-71k.tar.gz'
SPAIR_ZIP = '/content/drive/MyDrive/SPair-71k.tar.gz'
# ─────────────────────────────────────────────────────────────────────────

print(f'Looking for: {SPAIR_ZIP}')
print(f'Exists: {os.path.exists(SPAIR_ZIP)}')


---
## Cell 3 — Unzip SPair-71k


In [ ]:
import tarfile, zipfile, os

EXTRACT_DIR = '/content/SPair-71k'
SPAIR_ROOT  = EXTRACT_DIR

if os.path.exists(SPAIR_ROOT) and any(os.scandir(SPAIR_ROOT)):
    print(f'Already extracted at {SPAIR_ROOT}')
else:
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    print(f'Extracting {SPAIR_ZIP} ...')
    if SPAIR_ZIP.endswith('.tar.gz') or SPAIR_ZIP.endswith('.tgz'):
        with tarfile.open(SPAIR_ZIP, 'r:gz') as tar:
            tar.extractall(EXTRACT_DIR)
    elif SPAIR_ZIP.endswith('.tar'):
        with tarfile.open(SPAIR_ZIP, 'r:') as tar:
            tar.extractall(EXTRACT_DIR)
    elif SPAIR_ZIP.endswith('.zip'):
        with zipfile.ZipFile(SPAIR_ZIP, 'r') as z:
            z.extractall(EXTRACT_DIR)
    print('Done!')

# Auto-detect if SPair extracted into a subdirectory
for sub in os.listdir(EXTRACT_DIR):
    candidate = os.path.join(EXTRACT_DIR, sub)
    if os.path.isdir(candidate) and 'PairAnnotation' in os.listdir(candidate):
        SPAIR_ROOT = candidate
        break

print(f'SPair root: {SPAIR_ROOT}')
for d in ['JPEGImages', 'PairAnnotation']:
    exists = os.path.exists(os.path.join(SPAIR_ROOT, d))
    print(f'  {"OK" if exists else "MISSING"}: {d}')


---
## Cell 4 — Install Dependencies


In [ ]:
!pip install -q tqdm Pillow
print('Dependencies ready')


---
## Cell 5 — Dataset Loader


In [ ]:
import json
from pathlib import Path
import numpy as np
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def get_transform(image_size=840):
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


class SPairDataset(Dataset):
    def __init__(self, root, split='test', category=None, image_size=840, transform=None):
        self.root       = Path(root)
        self.image_size = image_size
        self.transform  = transform
        pair_ann_dir = self.root / 'PairAnnotation' / split
        if not pair_ann_dir.exists():
            raise FileNotFoundError(f'PairAnnotation not found at {pair_ann_dir}')
        self.pairs = []
        for p in sorted(pair_ann_dir.glob('*.json')):
            with open(p) as f:
                ann = json.load(f)
            if category and ann['category'] != category:
                continue
            self.pairs.append(ann)
        print(f'Loaded {len(self.pairs)} pairs (split={split}, category={category or "all"})')

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        ann = self.pairs[idx]
        cat = ann['category']
        src = Image.open(self.root / 'JPEGImages' / cat / ann['src_imname']).convert('RGB')
        trg = Image.open(self.root / 'JPEGImages' / cat / ann['trg_imname']).convert('RGB')
        sw, sh = src.size
        tw, th = trg.size
        src_kps = np.array(ann['src_kps'], dtype=np.float32).reshape(-1, 2)
        trg_kps = np.array(ann['trg_kps'], dtype=np.float32).reshape(-1, 2)
        mask    = np.array(ann.get('kps_used', [1]*len(src_kps)), dtype=bool)
        src_kps, trg_kps = src_kps[mask], trg_kps[mask]
        S = self.image_size
        src = src.resize((S, S), Image.BILINEAR)
        trg = trg.resize((S, S), Image.BILINEAR)
        src_kps = src_kps * [S/sw, S/sh]
        trg_kps = trg_kps * [S/tw, S/th]
        if self.transform:
            src = self.transform(src)
            trg = self.transform(trg)
        return dict(src_image=src, trg_image=trg,
                    src_kps=src_kps, trg_kps=trg_kps,
                    category=cat, pair_id=ann.get('pair_id', str(idx)))


print('SPairDataset ready')


---
## Cell 6 — DINOv2 Feature Extractor


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class DINOv2Extractor(nn.Module):
    VARIANTS = {
        'vits': ('dinov2_vits14', 384),
        'vitb': ('dinov2_vitb14', 768),
        'vitl': ('dinov2_vitl14', 1024),
        'vitg': ('dinov2_vitg14', 1536),
    }
    def __init__(self, variant='vitb', device='cuda'):
        super().__init__()
        self.device     = device
        self.patch_size = 14
        hub_name, self.feat_dim = self.VARIANTS[variant]
        print(f'Loading {hub_name} ...')
        self.model = torch.hub.load('facebookresearch/dinov2', hub_name, pretrained=True)
        self.model.eval().to(device)
        for p in self.model.parameters():
            p.requires_grad_(False)
        print(f'DINOv2-{variant} loaded | patch_size={self.patch_size} feat_dim={self.feat_dim}')

    def extract(self, x):
        x = x.to(self.device)
        B, _, H, W = x.shape
        h, w = H // self.patch_size, W // self.patch_size
        with torch.no_grad():
            out = self.model.get_intermediate_layers(x, n=1)[0]
        return out[:, 1:, :].reshape(B, h, w, self.feat_dim)

    def unfreeze_last_n_layers(self, n):
        for block in list(self.model.blocks)[-n:]:
            for p in block.parameters():
                p.requires_grad_(True)
        print(f'Unfroze last {n} blocks for fine-tuning')


print('DINOv2Extractor ready')


---
## Cell 7 — Cosine Similarity Matcher


In [ ]:
def kp_to_patch(kp_xy, image_size, patch_size):
    n   = image_size // patch_size
    col = (kp_xy[:, 0] / patch_size).long().clamp(0, n-1)
    row = (kp_xy[:, 1] / patch_size).long().clamp(0, n-1)
    return torch.stack([row, col], dim=1)


def patch_to_kp(patch_rc, patch_size):
    x = patch_rc[:, 1].float() * patch_size + patch_size / 2.0
    y = patch_rc[:, 0].float() * patch_size + patch_size / 2.0
    return torch.stack([x, y], dim=1)


def predict_argmax(src_feat, trg_feat, src_kps, image_size, patch_size):
    h, w, C = src_feat.shape
    dev     = src_feat.device
    src_rc  = kp_to_patch(src_kps.to(dev), image_size, patch_size)
    sv      = src_feat[src_rc[:, 0], src_rc[:, 1], :]
    tf      = trg_feat.reshape(-1, C)
    sim     = F.normalize(sv, dim=-1) @ F.normalize(tf, dim=-1).T
    best    = sim.argmax(dim=-1)
    best_rc = torch.stack([best // w, best % w], dim=1)
    return patch_to_kp(best_rc, patch_size)


def predict_window_soft_argmax(src_feat, trg_feat, src_kps, image_size, patch_size, window_size=5):
    h, w, C = src_feat.shape
    dev     = src_feat.device
    half    = window_size // 2
    src_rc  = kp_to_patch(src_kps.to(dev), image_size, patch_size)
    sv      = src_feat[src_rc[:, 0], src_rc[:, 1], :]
    tf      = trg_feat.reshape(-1, C)
    sim     = F.normalize(sv, dim=-1) @ F.normalize(tf, dim=-1).T
    sim_map = sim.reshape(-1, h, w)
    preds   = []
    for i in range(sim_map.shape[0]):
        s       = sim_map[i]
        peak    = s.argmax()
        pr, pc  = (peak // w).item(), (peak % w).item()
        r0, r1  = max(0, pr-half), min(h, pr+half+1)
        c0, c1  = max(0, pc-half), min(w, pc+half+1)
        win     = s[r0:r1, c0:c1]
        wts     = F.softmax(win.reshape(-1), dim=0).reshape(win.shape)
        rows    = torch.arange(r0, r0+wts.shape[0], device=dev).float()
        cols    = torch.arange(c0, c0+wts.shape[1], device=dev).float()
        sr      = (wts.sum(dim=1) * rows).sum()
        sc      = (wts.sum(dim=0) * cols).sum()
        preds.append(torch.stack([sc*patch_size + patch_size/2, sr*patch_size + patch_size/2]))
    return torch.stack(preds)


print('Matcher functions ready')


---
## Cell 8 — PCK Evaluator


In [ ]:
from collections import defaultdict


class PCKAccumulator:
    def __init__(self, thresholds=(0.05, 0.10, 0.20)):
        self.T = thresholds
        self.reset()

    def reset(self):
        self.kp  = {t: [] for t in self.T}
        self.img = {t: [] for t in self.T}
        self.cat = defaultdict(lambda: {t: [] for t in self.T})

    def update(self, pred, gt, image_size, category=None):
        dist = torch.norm(pred.float() - gt.float().to(pred.device), dim=-1)
        for t in self.T:
            c = (dist <= t * image_size).cpu().tolist()
            self.kp[t].extend(c)
            self.img[t].append(float(np.mean(c)))
            if category:
                self.cat[category][t].extend(c)

    def summarise(self):
        return {
            'per_keypoint': {t: float(np.mean(self.kp[t]))*100  for t in self.T},
            'per_image':    {t: float(np.mean(self.img[t]))*100 for t in self.T},
            'per_category': {cat: {t: float(np.mean(v[t]))*100 for t in self.T}
                             for cat, v in self.cat.items()},
            'n_pairs':      len(self.img[self.T[0]]),
            'n_keypoints':  len(self.kp[self.T[0]]),
        }

    def print_summary(self, name=''):
        r  = self.summarise()
        hdr = f"{'':22}" + '  '.join(f'PCK@{t:.2f}' for t in self.T)
        sep = '-' * max(60, len(hdr))
        print(f'\n{sep}\n  {name}\n{sep}')
        print(hdr)
        print(sep)
        print(f"{'Per-keypoint':<22}" + '  '.join(f"{r['per_keypoint'][t]:>8.2f}%" for t in self.T))
        print(f"{'Per-image':<22}"    + '  '.join(f"{r['per_image'][t]:>8.2f}%"    for t in self.T))
        print(sep)
        if r['per_category']:
            print('\nPer-category PCK@0.10:')
            for cat in sorted(r['per_category']):
                print(f'  {cat:>20}: {r["per_category"][cat].get(0.10, 0):.2f}%')
        print(f'\nPairs: {r["n_pairs"]}  |  Keypoints: {r["n_keypoints"]}')


print('PCKAccumulator ready')


---
## Cell 9 — Config & Load Model


In [ ]:
# ── CONFIG (edit these) ──────────────────────────────────────────────────
BACKBONE       = 'dinov2'   # 'dinov2' only for now; 'sam' requires extra install
DINOV2_VARIANT = 'vitb'     # 'vits' (fast) | 'vitb' (balanced) | 'vitl' | 'vitg'
IMAGE_SIZE     = 840        # must be divisible by 14 (patch size)
SPLIT          = 'test'
CATEGORY       = None       # None = all categories; or e.g. 'cat', 'dog'
DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'
# ─────────────────────────────────────────────────────────────────────────

assert IMAGE_SIZE % 14 == 0, f'IMAGE_SIZE must be divisible by 14, got {IMAGE_SIZE}'

extractor = DINOv2Extractor(variant=DINOV2_VARIANT, device=DEVICE)
print(f'Device: {DEVICE} | Backbone: {BACKBONE}-{DINOV2_VARIANT} | Image size: {IMAGE_SIZE}')


---
## Cell 10 — Load Dataset


In [ ]:
transform = get_transform(image_size=IMAGE_SIZE)

dataset = SPairDataset(
    root=SPAIR_ROOT,
    split=SPLIT,
    category=CATEGORY,
    image_size=IMAGE_SIZE,
    transform=transform,
)

# Quick sanity check
s = dataset[0]
print(f'Category : {s["category"]}')
print(f'src_image: {s["src_image"].shape}')
print(f'src_kps  : {s["src_kps"].shape}  (N keypoints)')


---
## Cell 11 — Run Evaluation


In [ ]:
from tqdm import tqdm

acc = PCKAccumulator(thresholds=(0.05, 0.10, 0.20))
extractor.eval()

for sample in tqdm(dataset, desc=f'{BACKBONE}-{DINOV2_VARIANT} argmax'):
    src_t = sample['src_image'].unsqueeze(0).to(DEVICE)
    trg_t = sample['trg_image'].unsqueeze(0).to(DEVICE)

    src_feat = extractor.extract(src_t)[0]
    trg_feat = extractor.extract(trg_t)[0]

    src_kps = torch.tensor(sample['src_kps'], dtype=torch.float32)
    gt_kps  = torch.tensor(sample['trg_kps'], dtype=torch.float32)

    pred_kps = predict_argmax(
        src_feat, trg_feat, src_kps,
        image_size=IMAGE_SIZE,
        patch_size=extractor.patch_size,
    )
    acc.update(pred_kps, gt_kps, IMAGE_SIZE, category=sample['category'])

acc.print_summary(name=f'{BACKBONE}-{DINOV2_VARIANT} | argmax | size={IMAGE_SIZE}')


---
## Cell 12 — Save Results


In [ ]:
import json, os, shutil

os.makedirs('/content/results', exist_ok=True)
results = acc.summarise()
results['config'] = dict(backbone=BACKBONE, variant=DINOV2_VARIANT,
                         image_size=IMAGE_SIZE, split=SPLIT)

local_path = f'/content/results/{BACKBONE}_{DINOV2_VARIANT}_{SPLIT}_argmax.json'
with open(local_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Saved locally : {local_path}')

drive_path = f'/content/drive/MyDrive/{BACKBONE}_{DINOV2_VARIANT}_{SPLIT}_argmax.json'
shutil.copy(local_path, drive_path)
print(f'Saved to Drive: {drive_path}')


---
## Cell 13 — Visualise Predictions


In [ ]:
import matplotlib.pyplot as plt
import torch

IMAGENET_MEAN_T = torch.tensor(IMAGENET_MEAN).view(3,1,1)
IMAGENET_STD_T  = torch.tensor(IMAGENET_STD).view(3,1,1)

def unnorm(t):
    return (t * IMAGENET_STD_T + IMAGENET_MEAN_T).clamp(0,1).permute(1,2,0).numpy()

SHOW = 3
fig, axes = plt.subplots(SHOW, 2, figsize=(12, 5*SHOW))

for i in range(SHOW):
    s        = dataset[i]
    src_t    = s['src_image'].unsqueeze(0).to(DEVICE)
    trg_t    = s['trg_image'].unsqueeze(0).to(DEVICE)
    sf       = extractor.extract(src_t)[0]
    tf       = extractor.extract(trg_t)[0]
    src_kps  = torch.tensor(s['src_kps'], dtype=torch.float32)
    gt_kps   = torch.tensor(s['trg_kps'], dtype=torch.float32)
    pred_kps = predict_argmax(sf, tf, src_kps, IMAGE_SIZE, extractor.patch_size)
    src_img  = unnorm(s['src_image'])
    trg_img  = unnorm(s['trg_image'])
    ax_s, ax_t = axes[i]
    ax_s.imshow(src_img)
    ax_s.scatter(src_kps[:,0], src_kps[:,1], c='lime', s=60, zorder=5)
    ax_s.set_title(f"Source — {s['category']}"); ax_s.axis('off')
    ax_t.imshow(trg_img)
    ax_t.scatter(gt_kps[:,0],          gt_kps[:,1],          c='lime', s=60, zorder=5, label='GT')
    ax_t.scatter(pred_kps[:,0].cpu(), pred_kps[:,1].cpu(),   c='red',  s=60, marker='x', linewidths=2, zorder=6, label='Predicted')
    for j in range(len(gt_kps)):
        ax_t.plot([gt_kps[j,0], pred_kps[j,0].cpu()], [gt_kps[j,1], pred_kps[j,1].cpu()], 'w-', lw=0.8, alpha=0.5)
    ax_t.set_title('Target  |  green=GT   red x=Predicted'); ax_t.axis('off')

plt.tight_layout()
plt.savefig('/content/results/visualisation.png', dpi=120, bbox_inches='tight')
plt.show()
